In [56]:
# Import necessary libraries
import json
import re
import logging
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm
from collections import Counter, defaultdict
import pandas as pd
from datetime import datetime
from pathlib import Path
import os

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [57]:
# Configuration settings
CONFIG = {
    "model_id": "meta-llama/Meta-Llama-3-8B",
    "use_quantization": True,
    "device": device,
    "max_new_tokens": 500,
    "temperature": 0.1,
    "input_file": "output_entities2.json",  # Update this path
    "output_dir": "lulc_extraction_output",
    "max_sentences": 5,  # Set to limit processing (e.g., 50)
}

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)

# Define valid LULC relations
VALID_RELATIONS = [
    "CHANGE_TO",
    "INCREASES_BY",
    "DECREASES_BY",
    "CAUSES",
    "LOCATED_IN",
    "OCCURS_DURING",
    "MEASURES",
    "AFFECTS",
    "FROM_TO",
    "ENABLES"
]

print("Configuration loaded successfully")

Configuration loaded successfully


In [58]:
import os
import json

# Check current working directory
print("Current working directory:", os.getcwd())

# List all files in current directory
print("\nAll files in current directory:")
for file in os.listdir('.'):
    print(f"  📄 {file}")

# Specifically check for your file
file_name = 'output_entities2.json'
if os.path.exists(file_name):
    print(f"\n✅ Found {file_name}")
    # Check file size to make sure it's not empty
    file_size = os.path.getsize(file_name)
    print(f"File size: {file_size} bytes")
else:
    print(f"\n❌ {file_name} not found in current directory")

Current working directory: /home/raham/ARENA 2025

All files in current directory:
  📄 output_skweak_partial_voting
  📄 .gitignore
  📄 label_studio_tasks_with_ner_and_relations.json
  📄 enhanced_label_studio_config_20250613_154912.xml
  📄 papers
  📄 Untitled5.ipynb
  📄 label_studio_conf_xml.xml
  📄 your_data_label_studio.json
  📄 labelstudio_single_relation.json
  📄 flan_t5_extracted_lulc_events_refined_prompt.csv
  📄 LCprocess.csv
  📄 labelstudio_relation_extraction.json
  📄 grammar_label_studio_20250617_161106.json
  📄 enhanced_extraction_results
  📄 unique_lulc_terms.json
  📄 merged_file.json
  📄 PyPaperBot
  📄 output_hardcoded_enhanced_rules
  📄 LLaMA 3 Implementation for LULC Event Extraction.ipynb
  📄 output_context
  📄 extracted_text
  📄 context_aware_extraction.log
  📄 .git
  📄 processed_corpus_for_ner.csv
  📄 grobid
  📄 training_data_corrected.json
  📄 pre_annotated_label_studio.json
  📄 lulc_entities.csv
  📄 extracted_entities_structured.json
  📄 lulc_relationships.csv
  📄 ne

In [59]:
def load_processed_data(file_path):
    """Load already processed data (not raw Label Studio format)"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"✅ Loaded {len(data)} items from {file_path}")
        
        # Clean the sentence text (remove the "text': '" prefix)
        for item in data:
            sentence = item.get('sentence', '')
            if sentence.startswith("text': '"):
                sentence = sentence[8:]  # Remove "text': '"
            if sentence.endswith("'"):
                sentence = sentence[:-1]  # Remove trailing quote
            item['sentence'] = sentence
        
        # Print statistics
        total_entities = sum(len(item.get('entities', [])) for item in data)
        sentences_with_entities = sum(1 for item in data if item.get('entities', []))
        
        print(f"📊 Processing Statistics:")
        print(f"  - Total sentences: {len(data)}")
        print(f"  - Sentences with entities: {sentences_with_entities}")
        print(f"  - Total entities extracted: {total_entities}")
        print(f"  - Average entities per sentence: {total_entities/len(data):.2f}")
        
        return data
        
    except Exception as e:
        print(f"Error loading data: {e}")
        traceback.print_exc()
        return []

# Use the correct function for your data format
input_data = load_processed_data("output_entities2.json")

# Verify it worked
if input_data:
    print("\n🔍 First item check:")
    first_item = input_data[0]
    print(f"Sentence: {first_item['sentence'][:100]}...")
    print(f"Entities found: {len(first_item['entities'])}")
    if first_item['entities']:
        for ent in first_item['entities']:
            print(f"  - {ent['text']} ({ent['label']})")

✅ Loaded 67 items from output_entities2.json
📊 Processing Statistics:
  - Total sentences: 67
  - Sentences with entities: 67
  - Total entities extracted: 334
  - Average entities per sentence: 4.99

🔍 First item check:
Sentence: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often con...
Entities found: 2
  - the 1970s and 1980s (DATE)
  - loss (CHANGE)


In [60]:
def load_mistral_model(model_id, use_quantization=True):
    """Load Mistral model and tokenizer"""
    print(f"🔄 Loading model: {model_id}")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Configure quantization
        quantization_config = None
        if use_quantization and device == "cuda":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if device == "cuda" else None,
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        
        print(f"✅ Model loaded successfully on {device}")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

# Load the model
model, tokenizer = load_mistral_model(CONFIG["model_id"], CONFIG["use_quantization"])

🔄 Loading model: meta-llama/Meta-Llama-3-8B
🔧 Using 4-bit quantization


2025-07-10 10:55:30,686 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully on cuda


In [61]:
def build_lulc_extraction_prompt(sentence):
    """Build prompt for joint entity recognition and relation extraction"""
    
    prompt = f"""You are an expert in Land Use Land Cover (LULC) analysis. Perform joint entity recognition and relation extraction from the given sentence LETS DO THAT USING STEP BY STEP RESONING .

SENTENCE: "{sentence}"

**STRICT ENTITY TYPES (ONLY USE THESE - NO EXCEPTIONS):**
- CHANGE: Transformation words (increased, decreased, converted, expanded, reduced, grew, declined, lost, gained, loss)
- LOC: Location names (countries, cities, specific name of districts, specific name of provinces)
- LULC: Land Use/Land Cover types (forest, cropland, urban area, built-up area, grassland, wetland, water body, bare ground, agricultural land, residential area, degraded land, woody vegetation)
- DATE: Temporal references (years, months, periods, seasons, decades, 1970s, 1980s)
- PERCENT: Percentage values (25%, 10.5%, thirty percent)
- CARDINAL: Numeric values without % (1000, 2.5 million, three, 500 hectares)
- COORDINATES: Geographic coordinates (40.7°N, latitude 23.5)
- SURFACE_UNIT: Area measurements (100 hectares, 50 km², 1000 acres)
- PROCESS: Environmental processes (deforestation, urbanization, expansion, drought, flooding, desertification, degradation)
- QUANTITY: Other quantities (rate of change, annual loss, total area, large parts)
THERE IS NO ENTITY CALLED EVENT 
**ENTITY EXTRACTION RULES:**
1. Extract entities as concise as possible (e.g., "loss" not "observed loss of woody vegetation")
2. List each date separately (e.g., "1970s" and "1980s" as two entities)
3. "desertification" is a PROCESS (the process of becoming desert), not LULC
4. "woody vegetation" or "woody vegetation cover" is LULC
**STRICT RELATIONSHIP TYPES (ONLY USE THESE - NO EXCEPTIONS):**
**CHANGE_TO**: Indicates a direct transformation from one LULC type to another LULC type (2 diffrent LULC).
- ✅ CORRECT: forest --CHANGE_TO-- cropland (trees cut, land converted to farming)
- ✅ CORRECT: agricultural land --CHANGE_TO-- urban area (farmland developed into city)
- ✅ CORRECT: grassland --CHANGE_TO-- built-up area (grass removed, buildings constructed)
- ❌ WRONG: built-up area --CHANGE_TO-- built-up area (same type, just quantity change)
- ❌ WRONG: forest --CHANGE_TO-- forest (same type, just area change)

**INCREASES_BY/DECREASES_BY**: For quantitative changes within same LULC type (lulc and percentage )
- ✅ CORRECT: built-up area --INCREASES_BY-- 12.77% (more built-up area, not transformation)
- ✅ CORRECT: forest --DECREASES_BY-- 25% (less forest area, not transformation)

**Other Relations:**
- CAUSES: Process entity causes a change (deforestation --CAUSES-- forest loss)
- LOCATED_IN: Spatial relationships between one entity and loc (forest --LOCATED_IN-- Brazil)
- OCCURS_DURING: Temporal relationships between one entity and date (change --OCCURS_DURING-- 2018)
- MEASURES: Quantitative relationships between change and percentage or quality (12.77% --MEASURES-- increase)
- AFFECTS: Impact relationships between process and lulc they must be a real impact  (urbanization --AFFECTS-- forest)
- FROM_TO: Value changes between 2 persentage entity or 2 SURFACE_UNIT entity (52.88% --FROM_TO-- 65.5%)
- ENABLES: One process enables another process (deforestation --ENABLES-- urbanization)

**Relationship Types - BE VERY THOUGHTFUL:**
...
- OCCURS_DURING: Temporal relationships where a **change, process** takes place or is observed within a specific time period. (e.g., change --OCCURS_DURING-- 2018, urbanization --OCCURS_DURING-- decade)
❌ WRONG: simulation results --OCCURS_DURING-- study period (results don't 'occur' in time, they are 'from' or 'valid for' a period)

**CRITICAL THINKING RULES:**
1. **Ask yourself**: Is this ACTUALLY a transformation between different land types?
2. **Think about the process**: What physical change happened to the land?
3. **Consider causality**: What caused what? Don't create meaningless loops
4. **Be precise with measurements**: Percentages usually MEASURE changes, not cause them
5. **Temporal logic**: Changes happen DURING time periods, not TO time periods
6. **Spatial logic**: Things are LOCATED_IN places, places don't transform to places

INSTRUCTIONS:
1. Extract ONLY relations that are explicitly stated or directly implied in the sentence
2. Use ONLY the entities provided above
3. Each relation must include the entity label in the format: entity_text:ENTITY_LABEL
4. Each relation must follow the format: source_entity:SOURCE_LABEL --RELATIONSHIP-- target_entity:TARGET_LABEL
5. Include confidence level (HIGH/MEDIUM/LOW) for each relation
6. Do not create relations between entities of the same type using TRANSFORMS_TO

**OUTPUT FORMAT:**
ENTITIES:
- entity_text | ENTITY_TYPE

RELATIONS:
- entity:ENTITY_TYPE --RELATIONSHIP-- entity:ENTITY_TYPE | CONF: confidence_level

Now perform joint entity recognition and relation extraction:"""
    
    return prompt

In [62]:
def generate_relations(sentence, model, tokenizer):
    """Generate entities and relations using the model"""
    
    # Build prompt - now only needs sentence
    prompt = build_lulc_extraction_prompt(sentence)
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3000,
        padding=True
    )
    
    # Move to device
    if device == "cuda":
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    
    # Decode response
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response

# Test generation with first sentence
if input_data:
    test_response = generate_relations(
        input_data[0]['sentence'],
        model,
        tokenizer
    )
    print("Model response:")
    print(test_response)

Model response:
 

SENTENCE: "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel were designated as degraded land (e.g."




In [63]:
# Define valid types
VALID_ENTITY_TYPES = {
    'CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 'CARDINAL', 
    'COORDINATES', 'SURFACE_UNIT', 'PROCESS', 'QUANTITY'
}

VALID_RELATION_TYPES = {
    'CHANGE_TO', 'INCREASES_BY', 'DECREASES_BY', 'CAUSES', 
    'LOCATED_IN', 'OCCURS_DURING', 'MEASURES', 'AFFECTS', 
    'FROM_TO', 'ENABLES'
}

def parse_relations_from_response(response, sentence, log_all_relations=True):
    """Parse entities and relations from model response with strict filtering"""
    entities = []
    relations = []
    filtered_entities = 0
    filtered_relations = 0
    
    lines = response.strip().split('\n')
    in_entities_section = False
    in_relations_section = False
    
    # First, parse entities from the response
    entity_dict = {}  # Map entity text to its type
    
    for line in lines:
        line = line.strip()
        
        # Check sections
        if 'ENTITIES:' in line.upper():
            in_entities_section = True
            in_relations_section = False
            continue
        elif 'RELATIONS:' in line.upper():
            in_relations_section = True
            in_entities_section = False
            continue
            
        # Parse entities
        if in_entities_section and '|' in line:
            # Parse entity format: "- entity_text | ENTITY_TYPE"
            match = re.match(r'^-?\s*(.+?)\s*\|\s*([A-Z_]+)', line)
            if match:
                entity_text = match.group(1).strip()
                entity_type = match.group(2).strip()
                
                # FILTER: Check if entity type is valid
                if entity_type in VALID_ENTITY_TYPES:
                    entity_dict[entity_text] = entity_type
                    entities.append({
                        'text': entity_text,
                        'label': entity_type
                    })
                    if log_all_relations:
                        logging.info(f"✅ Valid Entity: {entity_text} | {entity_type}")
                else:
                    filtered_entities += 1
                    if log_all_relations:
                        logging.warning(f"❌ Filtered Invalid Entity Type: {entity_text} | {entity_type}")
            
        # Parse relations
        if in_relations_section and '--' in line:
            # Parse format: "entity:TYPE --RELATIONSHIP-- entity:TYPE | CONF: level"
            patterns = [
                r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)-->\s*(.+?):([A-Z_]+)\s*\|\s*CONF:\s*(\w+)',
                r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)--\s*(.+?):([A-Z_]+)\s*\|\s*CONF:\s*(\w+)',
                r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)--\s*(.+?):([A-Z_]+)',
            ]
            
            for pattern in patterns:
                match = re.match(pattern, line)
                if match:
                    source_text = match.group(1).strip()
                    source_type = match.group(2).strip()
                    relationship = match.group(3).strip()
                    target_text = match.group(4).strip()
                    target_type = match.group(5).strip()
                    confidence = match.group(6).strip().upper() if len(match.groups()) >= 6 else "MEDIUM"
                    
                    # FILTER: Check if relationship type is valid
                    if relationship in VALID_RELATION_TYPES:
                        # FILTER: Also check if entity types are valid
                        if source_type in VALID_ENTITY_TYPES and target_type in VALID_ENTITY_TYPES:
                            relation = {
                                'source': source_text,
                                'source_type': source_type,
                                'relationship': relationship,
                                'target': target_text,
                                'target_type': target_type,
                                'confidence': confidence
                            }
                            relations.append(relation)
                            if log_all_relations:
                                logging.info(f"✅ Valid Relation: {source_text}:{source_type} --{relationship}-- {target_text}:{target_type}")
                        else:
                            filtered_relations += 1
                            if log_all_relations:
                                logging.warning(f"❌ Filtered - Invalid entity types in relation: {source_type}, {target_type}")
                    else:
                        filtered_relations += 1
                        if log_all_relations:
                            logging.warning(f"❌ Filtered Invalid Relationship Type: {relationship}")
                    break
    
    if log_all_relations and (filtered_entities > 0 or filtered_relations > 0):
        logging.info(f"📊 Filtering Summary: Filtered {filtered_entities} invalid entities and {filtered_relations} invalid relations")
    
    return {
        'entities': entities,
        'relations': relations,
        'filtered_entities': filtered_entities,
        'filtered_relations': filtered_relations
    }

In [64]:
# Test generation with first sentence
if input_data:
    test_response = generate_relations(
        input_data[0]['sentence'],
        model,
        tokenizer
    )
    print("Model response:")
    print(test_response)
    
    # Parse the response
    parsed_result = parse_relations_from_response(test_response, input_data[0]['sentence'])
    print("\nParsed Entities:")
    for entity in parsed_result['entities']:
        print(f"  - {entity['text']} | {entity['label']}")
    print("\nParsed Relations:")
    for rel in parsed_result['relations']:
        print(f"  - {rel['source']}:{rel['source_type']} --{rel['relationship']}-- {rel['target']}:{rel['target_type']} | CONF: {rel['confidence']}")

Model response:
 

SENTENCE: "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel were designated as degraded land (e.g."



Parsed Entities:

Parsed Relations:


In [65]:
def process_all_sentences(input_data, model, tokenizer, max_sentences=None):
    """Process all sentences to extract entities and relations with improved error handling"""
    
    # Limit processing if specified
    if max_sentences:
        input_data = input_data[:max_sentences]
        print(f"🔄 Processing {max_sentences} sentences (limited for testing)")
    else:
        print(f"🔄 Processing all {len(input_data)} sentences")
    
    all_results = []
    successful_extractions = 0
    failed_extractions = 0
    total_entities_extracted = 0
    total_relations_extracted = 0
    
    # Process each sentence
    for idx, item in enumerate(tqdm(input_data, desc="Extracting entities and relations")):
        sentence = item['sentence']
        
        # Clean sentence if needed
        if sentence.startswith("text': '"):
            sentence = sentence[8:]
        if sentence.endswith("'"):
            sentence = sentence[:-1]
        
        try:
            # Generate entities and relations
            response = generate_relations(sentence, model, tokenizer)
            
            # Parse entities and relations from response
            parsed_result = parse_relations_from_response(response, sentence, log_all_relations=False)
            
            # Count extracted items
            num_entities = len(parsed_result['entities'])
            num_relations = len(parsed_result['relations'])
            
            # Determine if extraction was successful
            # Consider it successful if we have either entities OR relations
            is_successful = (num_entities > 0 or num_relations > 0)
            
            if is_successful:
                successful_extractions += 1
                total_entities_extracted += num_entities
                total_relations_extracted += num_relations
                logger.info(f"✅ Sentence {idx}: Found {num_entities} entities and {num_relations} relations")
            else:
                failed_extractions += 1
                logger.warning(f"⚠️ Sentence {idx}: No entities or relations extracted")
            
            # Store results
            result = {
                'sentence_id': idx,
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': parsed_result['entities'],
                'extracted_relations': parsed_result['relations'],
                'num_entities': num_entities,
                'num_relations': num_relations,
                'filtered_entities': parsed_result.get('filtered_entities', 0),
                'filtered_relations': parsed_result.get('filtered_relations', 0),
                'model_response': response,
                'processing_timestamp': datetime.now().isoformat(),
                'success': is_successful
            }
            
            all_results.append(result)
                
        except Exception as e:
            logger.error(f"❌ Error processing sentence {idx}: {e}")
            failed_extractions += 1
            
            # Store error result
            error_result = {
                'sentence_id': idx,
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': [],
                'extracted_relations': [],
                'num_entities': 0,
                'num_relations': 0,
                'filtered_entities': 0,
                'filtered_relations': 0,
                'model_response': f"ERROR: {str(e)}",
                'processing_timestamp': datetime.now().isoformat(),
                'success': False,
                'error': str(e)
            }
            all_results.append(error_result)
    
    # Calculate statistics
    sentences_with_entities = sum(1 for result in all_results if result['num_entities'] > 0)
    sentences_with_relations = sum(1 for result in all_results if result['num_relations'] > 0)
    sentences_with_both = sum(1 for result in all_results if result['num_entities'] > 0 and result['num_relations'] > 0)
    
    # Print comprehensive summary
    print(f"\n📊 Processing Complete!")
    print(f"{'='*50}")
    print(f"📈 Success Metrics:")
    print(f"  - Total sentences processed: {len(input_data)}")
    print(f"  - Successful extractions: {successful_extractions}")
    print(f"  - Failed extractions: {failed_extractions}")
    print(f"  - Success rate: {(successful_extractions/len(input_data)*100):.1f}%")
    
    print(f"\n📋 Entity Statistics:")
    print(f"  - Total entities extracted: {total_entities_extracted}")
    print(f"  - Sentences with entities: {sentences_with_entities}")
    print(f"  - Average entities per sentence: {total_entities_extracted/len(input_data):.2f}")
    if sentences_with_entities > 0:
        print(f"  - Average entities per successful sentence: {total_entities_extracted/sentences_with_entities:.2f}")
    
    print(f"\n🔗 Relation Statistics:")
    print(f"  - Total relations extracted: {total_relations_extracted}")
    print(f"  - Sentences with relations: {sentences_with_relations}")
    print(f"  - Average relations per sentence: {total_relations_extracted/len(input_data):.2f}")
    if sentences_with_relations > 0:
        print(f"  - Average relations per successful sentence: {total_relations_extracted/sentences_with_relations:.2f}")
    
    print(f"\n🎯 Combined Statistics:")
    print(f"  - Sentences with both entities and relations: {sentences_with_both}")
    
    # Show filtering statistics if available
    total_filtered_entities = sum(result.get('filtered_entities', 0) for result in all_results)
    total_filtered_relations = sum(result.get('filtered_relations', 0) for result in all_results)
    if total_filtered_entities > 0 or total_filtered_relations > 0:
        print(f"\n🚫 Filtering Statistics:")
        print(f"  - Total entities filtered: {total_filtered_entities}")
        print(f"  - Total relations filtered: {total_filtered_relations}")
    
    return all_results

# Update CONFIG if needed
CONFIG = {
    "max_sentences": 2,  # Set to None for all sentences
    "output_dir": "lulc_extraction_output"
}

# Process all sentences with the improved function
print("🚀 Starting improved joint entity and relation extraction...")
results = process_all_sentences(
    input_data, 
    model, 
    tokenizer, 
    max_sentences=CONFIG["max_sentences"]
)

# Save results to JSON file
output_file = Path(CONFIG["output_dir"]) / f"lulc_joint_extraction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
output_file.parent.mkdir(exist_ok=True)  # Create directory if it doesn't exist

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"💾 Results saved to: {output_file}")

# Display sample results
print(f"\n🔍 Sample Results:")
print(f"{'='*50}")

successful_results = [r for r in results if r['success']]
for i, result in enumerate(successful_results[:3]):
    print(f"\n--- Sample {i+1} (Sentence ID: {result['sentence_id']}) ---")
    print(f"Sentence: {result['sentence'][:100]}...")
    
    if result['extracted_entities']:
        print(f"✅ Entities ({result['num_entities']}):")
        for ent in result['extracted_entities'][:5]:
            print(f"  • {ent['text']} | {ent['label']}")
        if result['num_entities'] > 5:
            print(f"  ... and {result['num_entities'] - 5} more entities")
    
    if result['extracted_relations']:
        print(f"🔗 Relations ({result['num_relations']}):")
        for rel in result['extracted_relations'][:3]:
            print(f"  ➤ {rel['source']}:{rel['source_type']} --{rel['relationship']}--> {rel['target']}:{rel['target_type']} | {rel['confidence']}")
        if result['num_relations'] > 3:
            print(f"  ... and {result['num_relations'] - 3} more relations")

if not successful_results:
    print("❌ No successful extractions found. Check your valid entity/relation types!")

🚀 Starting improved joint entity and relation extraction...
🔄 Processing 2 sentences (limited for testing)


Extracting entities and relations:   0%|          | 0/2 [00:00<?, ?it/s]

2025-07-10 10:55:47,052 - ERROR - ❌ Error processing sentence 0: 'max_new_tokens'
2025-07-10 10:55:47,057 - ERROR - ❌ Error processing sentence 1: 'max_new_tokens'



📊 Processing Complete!
📈 Success Metrics:
  - Total sentences processed: 2
  - Successful extractions: 0
  - Failed extractions: 2
  - Success rate: 0.0%

📋 Entity Statistics:
  - Total entities extracted: 0
  - Sentences with entities: 0
  - Average entities per sentence: 0.00

🔗 Relation Statistics:
  - Total relations extracted: 0
  - Sentences with relations: 0
  - Average relations per sentence: 0.00

🎯 Combined Statistics:
  - Sentences with both entities and relations: 0
💾 Results saved to: lulc_extraction_output/lulc_joint_extraction_20250710_105547.json

🔍 Sample Results:
❌ No successful extractions found. Check your valid entity/relation types!


In [15]:
# Test with the working approach from earlier
test_sentence = input_data[0]['sentence']

# Clean the sentence properly
if test_sentence.startswith("text': '"):
    test_sentence = test_sentence[8:]
if test_sentence.endswith("'"):
    test_sentence = test_sentence[:-1]

print(f"Testing with cleaned sentence: {test_sentence[:100]}...")

# Generate and parse
test_response = generate_relations(test_sentence, model, tokenizer)
parsed_result = parse_relations_from_response(test_response, test_sentence)

print(f"Results: {len(parsed_result['entities'])} entities, {len(parsed_result['relations'])} relations")

Testing with cleaned sentence: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often con...
Results: 0 entities, 0 relations


In [10]:
def save_extraction_results(results, output_dir):
    """Save results in multiple formats"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Compute basic statistics
    stats = {
        'total_sentences_processed': len(results),
        'sentences_with_entities': sum(1 for r in results if r.get('extracted_entities')),
        'sentences_with_relations': sum(1 for r in results if r.get('extracted_relations')),
        'total_entities_extracted': sum(len(r.get('extracted_entities', [])) for r in results),
        'total_relations_extracted': sum(len(r.get('extracted_relations', [])) for r in results),
        'entity_types': {},
        'relation_types': {}
    }
    
    # Count entity types
    for result in results:
        for entity in result.get('extracted_entities', []):
            entity_type = entity.get('label', 'UNKNOWN')
            stats['entity_types'][entity_type] = stats['entity_types'].get(entity_type, 0) + 1
    
    # Count relation types
    for result in results:
        for relation in result.get('extracted_relations', []):
            relation_type = relation.get('relationship', 'UNKNOWN')
            stats['relation_types'][relation_type] = stats['relation_types'].get(relation_type, 0) + 1
    
    # 1. Save detailed results
    detailed_output = {
        'metadata': {
            'extraction_date': datetime.now().isoformat(),
            'model': CONFIG['model_id'],
            'extraction_type': 'joint_entity_relation',
            'statistics': stats
        },
        'results': results
    }
    
    detailed_path = Path(output_dir) / f"joint_extraction_{timestamp}.json"
    with open(detailed_path, 'w', encoding='utf-8') as f:
        json.dump(detailed_output, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Detailed results saved to: {detailed_path}")
    
    # 2. Save entities CSV
    entity_csv_data = []
    for result in results:
        sentence_id = result.get('sentence_id', 'N/A')
        sentence = result.get('sentence', 'N/A')
        for entity in result.get('extracted_entities', []):
            entity_csv_data.append({
                'sentence_id': sentence_id,
                'sentence': sentence,
                'entity_text': entity['text'],
                'entity_type': entity['label']
            })
    
    if entity_csv_data:
        entity_df = pd.DataFrame(entity_csv_data)
        entity_csv_path = Path(output_dir) / f"entities_table_{timestamp}.csv"
        entity_df.to_csv(entity_csv_path, index=False)
        print(f"✅ Entities CSV saved to: {entity_csv_path}")
    
    # 3. Save relations CSV with entity types
    relation_csv_data = []
    for result in results:
        sentence_id = result.get('sentence_id', 'N/A')
        sentence = result.get('sentence', 'N/A')
        for relation in result.get('extracted_relations', []):
            relation_csv_data.append({
                'sentence_id': sentence_id,
                'sentence': sentence,
                'source': relation['source'],
                'source_type': relation.get('source_type', 'UNKNOWN'),
                'relationship': relation['relationship'],
                'target': relation['target'],
                'target_type': relation.get('target_type', 'UNKNOWN'),
                'confidence': relation.get('confidence', 'MEDIUM')
            })
    
    if relation_csv_data:
        relation_df = pd.DataFrame(relation_csv_data)
        relation_csv_path = Path(output_dir) / f"relations_table_{timestamp}.csv"
        relation_df.to_csv(relation_csv_path, index=False)
        print(f"✅ Relations CSV saved to: {relation_csv_path}")
    
    # 4. Save statistics
    stats_path = Path(output_dir) / f"statistics_{timestamp}.json"
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
    
    print(f"✅ Statistics saved to: {stats_path}")
    
    # 5. Create a summary report
    summary = f"""
=== Joint Entity & Relation Extraction Summary ===
Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Model: {CONFIG['model_id']}

Sentences processed: {stats['total_sentences_processed']}
Sentences with entities: {stats['sentences_with_entities']}
Sentences with relations: {stats['sentences_with_relations']}

Total entities extracted: {stats['total_entities_extracted']}
Total relations extracted: {stats['total_relations_extracted']}

Entity types distribution:
"""
    for entity_type, count in sorted(stats['entity_types'].items(), key=lambda x: x[1], reverse=True):
        summary += f"  - {entity_type}: {count}\n"
    
    summary += "\nRelation types distribution:\n"
    for rel_type, count in sorted(stats['relation_types'].items(), key=lambda x: x[1], reverse=True):
        summary += f"  - {rel_type}: {count}\n"
    
    summary_path = Path(output_dir) / f"summary_{timestamp}.txt"
    with open(summary_path, 'w', encoding='utf-8') as f:
        f.write(summary)
    
    print(f"✅ Summary report saved to: {summary_path}")
    
    return {
        'detailed_results': detailed_path,
        'entities_csv': entity_csv_path if entity_csv_data else None,
        'relations_csv': relation_csv_path if relation_csv_data else None,
        'statistics': stats_path,
        'summary': summary_path
    }

# Save all results
paths = save_extraction_results(results, CONFIG["output_dir"])

# Display summary statistics
print("\n📊 Extraction Summary:")
for path_type, path in paths.items():
    if path:
        print(f"  - {path_type}: {path}")

✅ Detailed results saved to: lulc_extraction_output/joint_extraction_20250709_140919.json
✅ Entities CSV saved to: lulc_extraction_output/entities_table_20250709_140919.csv
✅ Relations CSV saved to: lulc_extraction_output/relations_table_20250709_140919.csv
✅ Statistics saved to: lulc_extraction_output/statistics_20250709_140919.json
✅ Summary report saved to: lulc_extraction_output/summary_20250709_140919.txt

📊 Extraction Summary:
  - detailed_results: lulc_extraction_output/joint_extraction_20250709_140919.json
  - entities_csv: lulc_extraction_output/entities_table_20250709_140919.csv
  - relations_csv: lulc_extraction_output/relations_table_20250709_140919.csv
  - statistics: lulc_extraction_output/statistics_20250709_140919.json
  - summary: lulc_extraction_output/summary_20250709_140919.txt


In [11]:
def create_label_studio_output(results, original_data, output_dir):
    """Create Label Studio compatible output with relations - FIXED VERSION"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    label_studio_tasks = []
    
    for idx, (result, orig_item) in enumerate(zip(results, original_data)):
        # Start with predictions structure
        predictions = []
        entity_id_map = {}
        entity_counter = 0
        
        # Get the original text
        text = orig_item['original_data']['data']['text']
        
        # Step 1: Add entities from original annotations
        if 'annotations' in orig_item['original_data'] and orig_item['original_data']['annotations']:
            for annotation in orig_item['original_data']['annotations']:
                if 'result' in annotation:
                    for res in annotation['result']:
                        if res['type'] == 'labels':
                            entity_id = f"ent_{idx}_{entity_counter}"
                            entity_counter += 1
                            entity_id_map[res['value']['text']] = entity_id
                            
                            # Add entity prediction
                            entity_pred = res.copy()
                            entity_pred['id'] = entity_id
                            predictions.append(entity_pred)
        
        # Step 2: Check for entities from model extraction that might not be in annotations
        for entity in result.get('entities', []):
            if entity['text'] not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[entity['text']] = entity_id
                
                # Create new entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': entity['start_char'] if entity['start_char'] >= 0 else 0,
                        'end': entity['end_char'] if entity['end_char'] >= 0 else len(entity['text']),
                        'text': entity['text'],
                        'labels': [entity['label']]
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
        
        # Step 3: Process relations and create missing entities
        for rel_idx, relation in enumerate(result['relations']):
            source = relation['source']
            target = relation['target']
            
            # If source entity not found, create it
            if source not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[source] = entity_id
                
                # Try to find position in text
                source_start = text.lower().find(source.lower())
                source_end = source_start + len(source) if source_start != -1 else -1
                
                # Create entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': source_start if source_start >= 0 else 0,
                        'end': source_end if source_end >= 0 else len(source),
                        'text': source,
                        'labels': ['ENTITY']  # Default label for missing entities
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
                logger.info(f"Created new entity for relation source: {source}")
            
            # If target entity not found, create it
            if target not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[target] = entity_id
                
                # Try to find position in text
                target_start = text.lower().find(target.lower())
                target_end = target_start + len(target) if target_start != -1 else -1
                
                # Create entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': target_start if target_start >= 0 else 0,
                        'end': target_end if target_end >= 0 else len(target),
                        'text': target,
                        'labels': ['ENTITY']  # Default label for missing entities
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
                logger.info(f"Created new entity for relation target: {target}")
            
            # Now add the relation
            relation_pred = {
                'from_id': entity_id_map[source],
                'to_id': entity_id_map[target],
                'type': 'relation',
                'labels': [relation['relationship']],
                'direction': 'right',
                'meta': {
                    'confidence': relation['confidence']
                }
            }
            predictions.append(relation_pred)
        
        # Create Label Studio task
        task = {
            'data': orig_item['original_data']['data'],
            'predictions': [{
                'model_version': f'lulc_relations_{timestamp}',
                'result': predictions
            }]
        }
        
        # Keep original annotations if present
        if 'annotations' in orig_item['original_data']:
            task['annotations'] = orig_item['original_data']['annotations']
        
        label_studio_tasks.append(task)
    
    # Save Label Studio format
    ls_path = Path(output_dir) / f"label_studio_with_relations_{timestamp}.json"
    with open(ls_path, 'w', encoding='utf-8') as f:
        json.dump(label_studio_tasks, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Label Studio format saved to: {ls_path}")
    print(f"📊 Created {len(label_studio_tasks)} tasks with entities and relations")
    
    # Count statistics
    total_predictions = sum(len(task['predictions'][0]['result']) for task in label_studio_tasks)
    print(f"📊 Total predictions (entities + relations): {total_predictions}")
    
    # Create Label Studio config XML with all entity types
    config_xml = """<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
    <Label value="LULC" background="#FF6B6B"/>
    <Label value="DATE" background="#4ECDC4"/>
    <Label value="LOCATION" background="#45B7D1"/>
    <Label value="PERCENT" background="#96CEB4"/>
    <Label value="PROCESS" background="#FECA57"/>
    <Label value="QUANTITY" background="#FF9FF3"/>
    <Label value="CHANGE" background="#A55EEA"/>
    <Label value="ENTITY" background="#B4B4B4"/>
  </Labels>
  <Relations name="relation" toName="label">
    <Relation value="TRANSFORMS_TO" background="#FF6B6B"/>
    <Relation value="INCREASES_BY" background="#4ECDC4"/>
    <Relation value="DECREASES_BY" background="#45B7D1"/>
    <Relation value="CAUSES" background="#96CEB4"/>
    <Relation value="LOCATED_IN" background="#FECA57"/>
    <Relation value="OCCURS_DURING" background="#FF9FF3"/>
    <Relation value="MEASURES" background="#A55EEA"/>
    <Relation value="AFFECTS" background="#54A0FF"/>
    <Relation value="FROM_TO" background="#5F27CD"/>
    <Relation value="ENABLES" background="#00D2D3"/>
  </Relations>
</View>"""
    
    config_path = Path(output_dir) / "label_studio_config.xml"
    with open(config_path, 'w') as f:
        f.write(config_xml)
    
    print(f"✅ Label Studio config saved to: {config_path}")
    
    return ls_path

# Create Label Studio output with fixed function
ls_output_path = create_label_studio_output(results, input_data, CONFIG["output_dir"])

KeyError: 'relations'

In [ ]:
def create_minimal_test_file(output_dir):
    """Create a minimal test file to verify Label Studio import works"""
    
    # Minimal working example
    test_task = {
        "data": {
            "text": "The forest area decreased by 25% in Brazil during 2018."
        },
        "annotations": [{
            "id": "test_annotation_1",
            "completed_by": 1,
            "result": [
                {
                    "id": "entity_1",
                    "type": "labels",
                    "value": {
                        "start": 4,
                        "end": 15,
                        "text": "forest area",
                        "labels": ["LULC"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "id": "entity_2",
                    "type": "labels",
                    "value": {
                        "start": 29,
                        "end": 32,
                        "text": "25%",
                        "labels": ["PERCENT"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "id": "entity_3",
                    "type": "labels",
                    "value": {
                        "start": 36,
                        "end": 42,
                        "text": "Brazil",
                        "labels": ["LOCATION"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "from_id": "entity_1",
                    "to_id": "entity_2",
                    "type": "relation",
                    "labels": ["DECREASES_BY"],
                    "direction": "right"
                }
            ]
        }]
    }
    
    # Save minimal test
    test_path = Path(output_dir) / "minimal_test.json"
    with open(test_path, 'w') as f:
        json.dump([test_task], f, indent=2)
    
    print(f"✅ Minimal test file saved to: {test_path}")
    print("Try importing this file first!")
    
    return test_path

# Create minimal test
minimal_test_path = create_minimal_test_file(CONFIG["output_dir"])

In [13]:
import json
import logging
import traceback

logger = logging.getLogger(__name__) # Initialize logger

def load_data_from_custom_format(file_path):
    """
    Loads data from a custom JSON format where each item has 'sentence' and 'entities' keys.
    'entities' is expected to be a list of dicts with 'text', 'label', 'start', 'end'.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print(f"✅ Loaded {len(data)} items from {file_path}")

        processed_data = []

        for item in data:
            # Extract sentence text
            sentence = item.get('sentence', '')

            # Clean up the "text': '" prefix from the sentence if it exists
            # Based on your snippet, it seems to be 'text': '
            if sentence.startswith("text': '"):
                sentence = sentence[len("text': '"):]
            if sentence.endswith("'"): # Remove trailing quote if present
                sentence = sentence[:-1]

            # Directly access the 'entities' list
            raw_entities = item.get('entities', [])
            
            entities = []
            for ent in raw_entities:
                # Map 'start' to 'start_char' and 'end' to 'end_char' for consistency
                # with your previous function's expected output format.
                if all(k in ent for k in ['text', 'label', 'start', 'end']):
                    entities.append({
                        'text': ent['text'],
                        'label': ent['label'],
                        'start_char': ent['start'],
                        'end_char': ent['end']
                    })

            processed_data.append({
                'sentence': sentence,
                'entities': entities,
                'original_data': item # Keep original data for reference if needed
            })

        # Print statistics
        total_entities = sum(len(item['entities']) for item in processed_data)
        sentences_with_entities = sum(1 for item in processed_data if item['entities'])

        print(f"\n📊 Processing Statistics:")
        print(f"  - Total sentences: {len(processed_data)}")
        print(f"  - Sentences with entities: {sentences_with_entities}")
        print(f"  - Total entities extracted: {total_entities}")
        print(f"  - Average entities per sentence: {total_entities/len(processed_data):.2f}")

        return processed_data

    except Exception as e:
        logger.error(f"Error loading data: {e}")
        traceback.print_exc()
        return []

# --- Usage Example ---
# Assuming your file is named 'output_entities2.json' and is directly in the format you showed.
# Replace with your actual file path if different.
file_path = "output_entities2.json" 

input_data = load_data_from_custom_format(file_path)

# Verify the fix worked
if input_data:
    print("\n🔍 First item check:")
    first_item = input_data[0]
    print(f"Sentence: {first_item['sentence'][:100]}...") # Print first 100 chars
    print(f"Entities found: {len(first_item['entities'])}")
    if first_item['entities']:
        for ent in first_item['entities']:
            print(f"  - {ent['text']} ({ent['label']}) [Chars: {ent.get('start_char')}-{ent.get('end_char')}]")
    else:
        print("  No entities found for the first item after processing.")

✅ Loaded 67 items from output_entities2.json

📊 Processing Statistics:
  - Total sentences: 67
  - Sentences with entities: 67
  - Total entities extracted: 334
  - Average entities per sentence: 4.99

🔍 First item check:
Sentence: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often con...
Entities found: 2
  - the 1970s and 1980s (DATE) [Chars: 30-49]
  - loss (CHANGE) [Chars: 64-68]


In [14]:
def load_mistral_model(model_id, use_quantization=True):
    """Load Mistral model and tokenizer"""
    print(f"🔄 Loading model: {model_id}")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Configure quantization
        quantization_config = None
        if use_quantization and device == "cuda":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if device == "cuda" else None,
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        
        print(f"✅ Model loaded successfully on {device}")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

# Load the model
model, tokenizer = load_mistral_model(CONFIG["model_id"], CONFIG["use_quantization"])

🔄 Loading model: mistralai/Mistral-7B-Instruct-v0.2
🔧 Using 4-bit quantization


2025-07-09 16:36:06,008 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Model loaded successfully on cuda


Sample prompt (first 500 chars):
You are an expert in Land Use Land Cover (LULC) analysis. Extract ONLY the relations between entities from the given sentence.

SENTENCE: "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel were designated as degraded land (e.g"

ENTITIES FOUND:
- the 1970s and 1980s | DATE
- loss | CHANGE

**Relationship Types - BE VERY THOUGHTFUL:**

**CHANGE_TO**: Indicates a direct transformation from one LU...


In [24]:
VALID_ENTITY_TYPES = {
    'CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 'CARDINAL', 
    'COORDINATES', 'SURFACE_UNIT', 'PROCESS', 'QUANTITY',
    'LULC_TYPE', 'ADJECTIVE'  # Added new types from your example
}

VALID_RELATION_TYPES = {
    'CHANGE_TO', 'INCREASES_BY', 'DECREASES_BY', 'CAUSES', 
    'LOCATED_IN', 'OCCURS_DURING', 'MEASURES', 'AFFECTS', 
    'FROM_TO', 'ENABLES', 'DESIGNATED_AS'  # Added new relation type
}

def generate_relations(entities, model, tokenizer):
    """Ultra strict version - repeatedly emphasizes entity constraints"""
    
    entity_texts = [e['text'] for e in entities]
    entity_list_numbered = '\n'.join([f"{i+1}. {entity}" for i, entity in enumerate(entity_texts)])
    
    prompt = f"""RELATION EXTRACTION TASK

ENTITIES YOU MUST USE (NO OTHERS ALLOWED):
{entity_list_numbered}

⚠️ CRITICAL CONSTRAINTS ⚠️
1. Use ONLY the {len(entity_texts)} entities listed above
2. Use the EXACT text as written
3. DO NOT invent new entities
4. DO NOT modify entity names
5. DO NOT use synonyms or similar words
6. If no relations exist between PROVIDED entities, output "No relations found"

ALLOWED ENTITY TYPES: {', '.join(sorted(VALID_ENTITY_TYPES))}
ALLOWED RELATION TYPES: {', '.join(sorted(VALID_RELATION_TYPES))}

OUTPUT FORMAT:
RELATIONS:
- entity_from_list:TYPE --RELATION-- entity_from_list:TYPE | CONF: HIGH/MEDIUM/LOW

ENTITIES TO USE (REMINDER): {', '.join(entity_texts)}

Extract relations now using ONLY the entities above:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.01,  # Extremely low temperature
            do_sample=False,   # Greedy decoding for most deterministic output
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.replace(prompt, "").strip()
    
    return response

In [25]:
def process_all_sentences(input_data, model, tokenizer, max_sentences=2):
    """Process all sentences to extract LULC relations"""
    
    print(f"🔄 Starting processing...")
    print(f"📊 Input data length: {len(input_data)}")
    
    # Limit processing if specified
    if max_sentences:
        input_data = input_data[:max_sentences]
        print(f"🔄 Processing {max_sentences} sentences (limited for testing)")
    else:
        print(f"🔄 Processing all {len(input_data)} sentences")
    
    all_results = []
    successful_extractions = 0
    failed_extractions = 0
    
    # Track entity type statistics
    entity_type_stats = {}
    relation_type_stats = {}
    
    # Process each sentence
    for idx, item in enumerate(tqdm(input_data, desc="Extracting relations")):
        print(f"\n--- Processing sentence {idx} ---")
        
        sentence = item['sentence']
        entities = item['entities']
        
        print(f"Sentence: {sentence[:100]}...")
        print(f"Entities: {[e['text'] for e in entities]}")
        
        # Skip sentences without entities
        if not entities:
            print(f"⚠️ Skipping sentence {idx}: No entities found")
            continue
        
        try:
            print(f"🔄 Generating relations...")
            # Generate relations - UPDATED: removed sentence parameter
            response = generate_relations(entities, model, tokenizer)
            print(f"📝 Model response: {response[:200]}...")
            
            print(f"🔍 Parsing relations...")
            # Parse relations from response
            relations = parse_relations_from_response(response, sentence, entities)
            print(f"✅ Found {len(relations)} relations")
            
            # Print each relation
            for i, rel in enumerate(relations):
                print(f"  {i+1}. {rel['source']}:{rel.get('source_type', 'UNKNOWN')} "
                      f"--{rel['relationship']}--> {rel['target']}:{rel.get('target_type', 'UNKNOWN')} "
                      f"| {rel['confidence']}")
            
            # Update statistics
            for relation in relations:
                source_type = relation.get('source_type', 'UNKNOWN')
                target_type = relation.get('target_type', 'UNKNOWN')
                relation_type = relation.get('relationship', 'UNKNOWN')
                
                entity_type_stats[source_type] = entity_type_stats.get(source_type, 0) + 1
                entity_type_stats[target_type] = entity_type_stats.get(target_type, 0) + 1
                relation_type_stats[relation_type] = relation_type_stats.get(relation_type, 0) + 1
            
            # Store results
            result = {
                'sentence_id': idx,
                'sentence': sentence,
                'entities': entities,
                'model_response': response,
                'extracted_relations': relations,
                'num_relations': len(relations),
                'processing_timestamp': datetime.now().isoformat()
            }
            
            all_results.append(result)
            
            if relations:
                successful_extractions += 1
                print(f"✅ Sentence {idx}: SUCCESS - Found {len(relations)} relations")
            else:
                failed_extractions += 1
                print(f"⚠️ Sentence {idx}: No valid relations extracted")
                
        except Exception as e:
            print(f"❌ ERROR processing sentence {idx}: {e}")
            print(f"   Exception type: {type(e).__name__}")
            import traceback
            print(f"   Traceback: {traceback.format_exc()}")
            
            failed_extractions += 1
            
            # Store error result
            error_result = {
                'sentence_id': idx,
                'sentence': sentence,
                'entities': entities,
                'model_response': f"ERROR: {str(e)}",
                'extracted_relations': [],
                'num_relations': 0,
                'processing_timestamp': datetime.now().isoformat(),
                'error': str(e)
            }
            all_results.append(error_result)
    
    # Print final statistics
    print(f"\n📊 Processing Complete!")
    print(f"  - Total sentences processed: {len(input_data)}")
    print(f"  - Successful extractions: {successful_extractions}")
    print(f"  - Failed extractions: {failed_extractions}")
    
    if successful_extractions + failed_extractions > 0:
        success_rate = (successful_extractions/(successful_extractions+failed_extractions)*100)
        print(f"  - Success rate: {success_rate:.1f}%")
    
    total_relations = sum(len(result['extracted_relations']) for result in all_results)
    sentences_with_relations = sum(1 for result in all_results if result['extracted_relations'])
    
    print(f"  - Total relations extracted: {total_relations}")
    print(f"  - Sentences with relations: {sentences_with_relations}")
    if sentences_with_relations > 0:
        print(f"  - Average relations per successful sentence: {total_relations/sentences_with_relations:.2f}")
    
    if entity_type_stats:
        print(f"\n📈 Entity Type Distribution:")
        sorted_entity_types = sorted(entity_type_stats.items(), key=lambda x: x[1], reverse=True)
        for entity_type, count in sorted_entity_types:
            print(f"  - {entity_type}: {count}")
    
    if relation_type_stats:
        print(f"\n🔗 Relation Type Distribution:")
        sorted_relation_types = sorted(relation_type_stats.items(), key=lambda x: x[1], reverse=True)
        for relation_type, count in sorted_relation_types:
            print(f"  - {relation_type}: {count}")
    
    return all_results

In [26]:
def save_extraction_results(results, output_dir):
    """Save results in multiple formats"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Compute basic statistics
    stats = {
        'total_sentences_processed': len(results),
        'sentences_with_relations': sum(1 for r in results if r.get('extracted_relations')),
        'total_relations_extracted': sum(len(r.get('extracted_relations', [])) for r in results),
        'relation_types': {},
        'entity_types': {}
    }
    
    # Count relation types and entity types
    for result in results:
        for relation in result.get('extracted_relations', []):
            relation_type = relation.get('relationship', 'UNKNOWN')
            source_type = relation.get('source_type', 'UNKNOWN')
            target_type = relation.get('target_type', 'UNKNOWN')
            
            stats['relation_types'][relation_type] = stats['relation_types'].get(relation_type, 0) + 1
            stats['entity_types'][source_type] = stats['entity_types'].get(source_type, 0) + 1
            stats['entity_types'][target_type] = stats['entity_types'].get(target_type, 0) + 1
    
    # 1. Save detailed results
    detailed_output = {
        'metadata': {
            'extraction_date': datetime.now().isoformat(),
            'model': CONFIG['model_id'],
            'statistics': stats
        },
        'results': results
    }
    
    detailed_path = Path(output_dir) / f"relations_extracted_{timestamp}.json"
    with open(detailed_path, 'w', encoding='utf-8') as f:
        json.dump(detailed_output, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Detailed results saved to: {detailed_path}")
    
    # 2. Save simplified CSV format with source and target labels
    csv_data = []
    for result in results:
        sentence_id = result.get('sentence_id', 'N/A')
        sentence = result.get('sentence', 'N/A')
        for relation in result.get('extracted_relations', []):
            csv_data.append({
                'sentence_id': sentence_id,
                'sentence': sentence,
                'source': relation['source'],
                'source_label': relation.get('source_type', 'UNKNOWN'),
                'relationship': relation['relationship'],
                'target': relation['target'],
                'target_label': relation.get('target_type', 'UNKNOWN'),
                'confidence': relation.get('confidence', 'MEDIUM')
            })
    
    if csv_data:
        df = pd.DataFrame(csv_data)
        csv_path = Path(output_dir) / f"relations_table_{timestamp}.csv"
        df.to_csv(csv_path, index=False)
        print(f"✅ CSV table saved to: {csv_path}")
        
        # Print sample of CSV data
        print(f"\n📋 CSV Preview (first 5 rows):")
        print(df.head().to_string(index=False))
    
    # 3. Save statistics with entity type breakdown
    stats_path = Path(output_dir) / f"statistics_{timestamp}.json"
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
    
    print(f"✅ Statistics saved to: {stats_path}")
    
    # 4. Create a summary report
    summary_path = Path(output_dir) / f"extraction_summary_{timestamp}.txt"
    with open(summary_path, 'w', encoding='utf-8') as f:
        f.write(f"LULC Relation Extraction Summary\n")
        f.write(f"Generated: {datetime.now().isoformat()}\n")
        f.write(f"Model: {CONFIG['model_id']}\n")
        f.write("=" * 50 + "\n\n")
        
        f.write(f"OVERALL STATISTICS:\n")
        f.write(f"- Total sentences processed: {stats['total_sentences_processed']}\n")
        f.write(f"- Sentences with relations: {stats['sentences_with_relations']}\n")
        f.write(f"- Total relations extracted: {stats['total_relations_extracted']}\n")
        f.write(f"- Success rate: {(stats['sentences_with_relations']/stats['total_sentences_processed']*100):.1f}%\n\n")
        
        f.write(f"RELATION TYPE DISTRIBUTION:\n")
        sorted_relations = sorted(stats['relation_types'].items(), key=lambda x: x[1], reverse=True)
        for rel_type, count in sorted_relations:
            f.write(f"- {rel_type}: {count}\n")
        
        f.write(f"\nENTITY TYPE DISTRIBUTION:\n")
        sorted_entities = sorted(stats['entity_types'].items(), key=lambda x: x[1], reverse=True)
        for ent_type, count in sorted_entities:
            f.write(f"- {ent_type}: {count}\n")
    
    print(f"✅ Summary report saved to: {summary_path}")
    
    # Print detailed statistics
    print(f"\n📊 Extraction Statistics:")
    print(f"  - Total sentences: {stats['total_sentences_processed']}")
    print(f"  - Sentences with relations: {stats['sentences_with_relations']}")
    print(f"  - Total relations: {stats['total_relations_extracted']}")
    print(f"  - Success rate: {(stats['sentences_with_relations']/stats['total_sentences_processed']*100):.1f}%")
    
    if stats['relation_types']:
        print(f"\n🔗 Top Relation Types:")
        sorted_relations = sorted(stats['relation_types'].items(), key=lambda x: x[1], reverse=True)[:5]
        for rel_type, count in sorted_relations:
            print(f"  - {rel_type}: {count}")
    
    if stats['entity_types']:
        print(f"\n🏷️ Top Entity Types:")
        sorted_entities = sorted(stats['entity_types'].items(), key=lambda x: x[1], reverse=True)[:5]
        for ent_type, count in sorted_entities:
            print(f"  - {ent_type}: {count}")
    
    return detailed_path, csv_path if csv_data else None, stats_path, summary_path

# Save all results
print("💾 Saving extraction results...")
paths = save_extraction_results(results, CONFIG["output_dir"])

print(f"\n📁 All files saved to: {CONFIG['output_dir']}")
print("Files generated:")
for i, path in enumerate(paths):
    if path:
        file_types = ["Detailed JSON", "CSV Table", "Statistics", "Summary Report"]
        print(f"  {i+1}. {file_types[i]}: {Path(path).name}")

💾 Saving extraction results...
✅ Detailed results saved to: lulc_extraction_output/relations_extracted_20250709_164517.json
✅ CSV table saved to: lulc_extraction_output/relations_table_20250709_164517.csv

📋 CSV Preview (first 5 rows):
 sentence_id                                                                                                                                                                                                         sentence       source source_label  relationship           target target_label confidence
           0 After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible [desertification] and large parts of the Sahel were designated as degraded land (e.g         loss       CHANGE  DECREASES_BY woody_vegetation         LULC       HIGH
           0 After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible [desertification] and